In [1]:
import json
# Sample JSON file
json_file = "../../../../../../Library/Gems3/ReacDC.backup"

def read_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

def process_json(data):
    for item in data:
        print(f"Processing item with key: {item.get('key', [])}")
        for entry in item.get("dod", []):
            print(f"ID: {entry.get('id', 'N/A')}, Label: {entry.get('label', 'Unknown')}, Value: {entry.get('val', 'None')}")

def save_json(data, filename="formatted_references.json"):
    """Save formatted data to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [2]:
# Read and process JSON data
json_data = read_json(json_file+ ".json")
#process_json(json_data)
# Save formatted data to a JSON file
save_json(json_data, json_file+ ".json")

In [3]:
def get_V0r_second(obj):
    """
    Returns the second value of V0r[0], or None if missing.
    Expected structure:
        V0r = [[v0, v1, v2]]
    """
    dod = obj.get("dod", {})
    v0r = dod.get("V0r")

    if (
        isinstance(v0r, list)
        and len(v0r) > 0
        and isinstance(v0r[0], list)
        and len(v0r[0]) > 1
    ):
        return v0r[0][1]

    return None


def matches_pattern(value, pattern):
    """
    Generic deep equality check.
    Returns True if value == pattern.
    """
    return value == pattern

def phaseOf(obj):
    """
    Returns the phase code from key[0], or None if missing.
    Examples:
      'a' → aqueous
      'g' → gas
      's' → solid
    """
    key = obj.get("key", [])
    return key[0] if key else None


def update_TPcMod(obj, phase, old_tpcmod, new_tpcmod,
                  v0r_second=None, tpcn="TPcMod"):
    """
    Update TPcMod if:
      - object's phase matches `phase`
      - TPcMod equals `old_tpcmod`
      - if v0r_second is provided, V0r[0][1] must equal it
    """
    if phaseOf(obj) != phase:
        return

    dod = obj.get("dod", {})
    tpc = dod.get(tpcn)

    if tpc != old_tpcmod:
        return

    # Optional V0r second-position check
    if v0r_second is not None:
        if get_V0r_second(obj) != v0r_second:
            return

    # All conditions satisfied → apply update
    dod[tpcn] = new_tpcmod


def process_objects(data, rules, tpcn="TPcMod"):
    """
    rules = list of dicts:
      {
        "phase": "a",
        "old": [...],
        "new": [...],
        "v0r_second": 0   # optional
      }
    """
    for obj in data:
        for rule in rules:
            update_TPcMod(
                obj,
                phase=rule["phase"],
                old_tpcmod=rule["old"],
                new_tpcmod=rule["new"],
                v0r_second=rule.get("v0r_second"),
                tpcn=tpcn,
            )



In [4]:
rules_DC = [
    {
        "phase": "a",   # aqueous
        "old": [["C", "S", "C", " ", " ", " "]],
        "new": [["C", "S", "N", "N ", " ", " "]],
    },
    {
        "phase": "g",   # gas
        "old": [["C", "S", "N", " ", " ", " "]],
        "new": [["C", "S", "N", "N ", " ", " "]],
    },
    {
        "phase": "s",   # solid
        "old": [["C", "S", "C", " ", " ", " "]],
        "new": [["C", "S", "C", "N ", " ", " "]],
    },
    {
        "phase": "l",   # liquid
        "old": [["C", "S", "C", " ", " ", " "]],
        "new": [["C", "S", "N", "N ", " ", " "]],
    }
]

rules_rDC = [
    {
        "phase": "a",   # aqueous
        "old": [["K", "3", "C", "N", " ", " "]],
        "new": [["K", "3", "N", "N ", " ", " "]],
        "v0r_second": 0 # only match if V0r[0][1] == 0
    },
    {
        "phase": "s",   # solid
        "old": [["K", "3", "C", "N", " ", " "]],
        "new": [["K", "3", "N", "N ", " ", " "]],
    },
 #   {
 #       "phase": "s",   # solid
 #       "old": [["C", "S", "C", " ", " ", " "]],
 #       "new": [["C", "S", "C", "N ", " ", " "]],
 #   }
]

#process_objects(json_data, rules_DC)
process_objects(json_data, rules_rDC, "REcMod")


In [5]:
save_json(json_data, json_file+ "formatted.json")